# Step 8: re-train MuRIL and XLM-R without the broken Hindi labels

**Before running:** `Runtime -> Change runtime type -> T4 GPU`. Then `Runtime -> Run all`.
When the first cell asks, upload **`data/train_clean.csv` and `data/val_clean.csv`** from the project
(made by `8_clean_train.py`). Takes ~30-40 min. At the end, two files download:
`preds_muril_clean.csv` and `preds_xlmr_clean.csv`. Put them in `results/` and run `8_clean_train.py` again.

Everything is identical to `2_finetune_colab.ipynb` except the training data: the 1,114 train and 195 val
Hindi posts that match CONSTRAINT 2021 (whose labels are unrelated to the original annotation) are removed.
The test set is the same.

**What fine-tuning means:** MuRIL and XLM-R were already pre-trained on huge amounts of text
(in many languages) to understand words in context. We add one small classification layer on
top and train the whole thing a little more on our labelled posts, so it learns *this* task.

- **XLM-R** (`xlm-roberta-base`): Facebook/Meta, trained on 100 languages of web text.
- **MuRIL** (`google/muril-base-cased`): Google, trained on 17 Indian languages, **including
  transliterated text** (Hindi written in English letters), so it should handle Hinglish better.

In [ ]:
!pip install -q sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

BASE = "https://huggingface.co/datasets/pankajbiswas6/prism-hinglish-hate-speech/resolve/main/data"
from google.colab import files
files.upload()  # pick train_clean.csv and val_clean.csv
train, val = pd.read_csv("train_clean.csv"), pd.read_csv("val_clean.csv")
test = pd.read_csv(f"{BASE}/test.csv")
for df in (train, val, test):
    df["text"] = df["text"].astype(str)
test["id"] = test.index  # same ids as 1_baseline.py, so predictions line up
print(len(train), len(val), len(test))

## Training settings
- `max_length=128` tokens: posts longer than that are cut (almost all posts are shorter).
- learning rate `2e-5`, 3 epochs, batch size 32: standard values for fine-tuning BERT-style models.
- We check macro-F1 on the **validation** set after each epoch. The **test** set is only used once, at the end.

In [ ]:
class PostDataset(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.enc = tok(list(df["text"]), truncation=True, max_length=128)
        self.labels = list(df["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro")}

def score_by_language(df):
    out = {"all": f1_score(df.label, df.pred, average="macro")}
    for lang in ["english", "hindi", "hinglish"]:
        part = df[df.lang == lang]
        out[lang] = f1_score(part.label, part.pred, average="macro")
    return {k: round(v, 3) for k, v in out.items()}

def train_and_predict(hf_name, short_name):
    set_seed(42)
    tok = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModelForSequenceClassification.from_pretrained(hf_name, num_labels=2)
    args = TrainingArguments(
        output_dir=f"out_{short_name}", learning_rate=2e-5, num_train_epochs=3,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        weight_decay=0.01, eval_strategy="epoch", save_strategy="no",
        fp16=torch.cuda.is_available(), report_to="none", seed=42, logging_steps=100,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=PostDataset(train, tok), eval_dataset=PostDataset(val, tok),
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    trainer.train()

    logits = trainer.predict(PostDataset(test, tok)).predictions
    out = test.copy()
    out["pred"] = np.argmax(logits, axis=1)
    print(short_name, score_by_language(out))
    out[["id", "text", "lang", "label", "pred"]].to_csv(f"preds_{short_name}.csv", index=False)

    del model, trainer
    torch.cuda.empty_cache()

In [ ]:
train_and_predict("google/muril-base-cased", "muril_clean")

In [ ]:
train_and_predict("xlm-roberta-base", "xlmr_clean")

## Download the predictions, then copy them into `results/` on your laptop

In [ ]:
from google.colab import files
files.download("preds_muril_clean.csv")
files.download("preds_xlmr_clean.csv")